# Spin and reservoir dynamics of polariton condensates

M0: scalar reference; M1: active/inactive reservoirs; M2: circular components with spin flip; M3: internal Josephson coupling.
The comparisons are recalculated for this report. Units: µm, ps, meV.

[Results](docs/results_overview.md) · [Methods](docs/methods.md) · [Checks](docs/diagnostic_summary.md)

In [ ]:
from pathlib import Path
from IPython.display import display, Image
from model import load_config, parameters
from numerics import solve_0d, observables_0d

config = load_config("F1", time={"final": 200.0})
p = parameters(config)
print("Threshold [µm⁻² ps⁻¹]:", p["Pth"])
print("Inactive-reservoir response [ps]:", p["tauI"])
example = solve_0d(config)
print("Demo final density [µm⁻²]:", observables_0d(example)["nc"][-1])

## Saved report figures N1–N6

Simplified main figures are in `results/figures/report/`: reservoir switching, circular populations, spatial components, density dynamics, relative phase, and a coupling summary. N3 appears when its new 2D series finishes. All are PNG, 160 mm wide at 300 dpi. English captions and measured results are in `docs/results_overview.md`. Original F1–F5 and D1–D3 remain supplementary.

In [ ]:
figures = sorted(Path("results/figures/report").glob("N*.png"))
for path in figures:
    print(path)
    display(Image(filename=str(path), width=800))
if not any(path.name.startswith("N3_") for path in figures):
    print("N3: spatial spin calculations are still pending; see results/data/N3_summary.json.")

## Recompute the homogeneous series

F1: empty reservoirs and pump switching. F3: 21 independent pump polarizations. F4/F5: 105 independent J values, with dense ranges near phase changes and the polarization reversal. Phase-masked cycles remain explicitly marked.

In [ ]:
DO_0D = False  # Set force=True in run_0d_series to recompute cached cases.
if DO_0D:
    from numerics import run_0d_series, run_0d_checks, run_phase_seed_checks, run_rabi_check
    run_0d_series(force=True)
    run_0d_checks()
    run_phase_seed_checks()
    run_rabi_check()

## Refine only the two changes in F5
Dense J ranges from config.yaml; the main window is 16000–18000 ps. Weak cycles and slow decay are checked up to 72000/144000 ps; the slowest case reaches 288000 ps. Cached states are reused. Optional controls test two attractors near the polarization reversal and phase sampling at 0.05 ps.

In [ ]:
DO_REFINEMENT = False
if DO_REFINEMENT:
    from numerics import (run_spin_refinement, run_refinement_checks,
                         run_polarization_controls, run_phase_sampling_checks)
    refined = run_spin_refinement()
    run_polarization_controls()
    run_phase_sampling_checks()
    run_refinement_checks()
    print([(r["J"], r["status"]) for r in refined if r.get("supplemental")])

## New spatial spin series N3

Four independent starts: pump polarization 0 and 0.3 on 512², plus 256² and half-step checks for 0.3. Same continuous equal-component seed, empty reservoirs, fixed total pump, J=0. Each component and its reservoirs must settle in two consecutive 100 ps windows. Runs resume checkpoints and extend together in 100 ps increments, up to 4000 ps if needed. Four worker processes are supported; one hour is a runtime guide, not a hard cutoff.

In [ ]:
DO_SPATIAL_SPIN = False
if DO_SPATIAL_SPIN:
    from numerics import run_spatial_spin_suite
    from plotting import make_report_figures
    spatial_results = run_spatial_spin_suite(max_workers=4)
    print(spatial_results["assessment"])
    print(make_report_figures())

## Recompute the ring comparison

This is the expensive part: three new 512² runs to 2000 ps and R2 grid/time comparisons. Saved checkpoints permit continuation. If settling is insufficient, the common comparison time is extended to 4000 ps.

In [ ]:
DO_RING = False
if DO_RING:
    from numerics import run_ring_suite
    ring_results = run_ring_suite(max_workers=1)
    print(ring_results["status"], ring_results.get("assessment"))

## Redraw from arrays

No time integration is performed by the figure exporter. Diagnostic sheets stay outside the report.

In [ ]:
if DO_0D or DO_RING or DO_REFINEMENT:
    from numerics import export_report_data
    export_report_data()

DO_REDRAW = False
if DO_REDRAW:
    from plotting import make_report_figures
    print(make_report_figures())

## Threshold selection for report 3 (Intel Arc)
Queue: coarse linear thresholds → double-precision checks → six short nonlinear starts.
Spot and ring each use 1.2 times their largest verified threshold upper bound;
B0/A1/T0 share the pump within a geometry. Stops before long production.
The launch switch is disabled by default. Status: results/data/report3/threshold_selection.json.


In [ ]:
DO_TETM_PILOTS = False
if DO_TETM_PILOTS:
    import subprocess
    log_path = Path("results/runs/report3/pilot.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a") as log:
        process = subprocess.Popen(["bash", "local/run_report3.sh"],
                                   stdout=log, stderr=subprocess.STDOUT,
                                   start_new_session=True)
    print("Pilot launcher PID:", process.pid, "Log:", log_path)

import json
pilot_summary = Path("results/data/report3/threshold_selection.json")
if pilot_summary.exists():
    summary = json.loads(pilot_summary.read_text())
    print("Pilot stage:", summary["status"], "Common p:", summary.get("selected_p"))
    for case, row in summary.get("startups", {}).items():
        print(case, row["time_ps"], "ps", row["status"])
for progress in sorted(Path("results/runs/report3").glob("near_threshold_*/progress.json")):
    print(progress.parent.name, json.loads(progress.read_text()))


## Short physical checks

One compact test file; explicit manual execution.

In [ ]:
DO_CHECKS = False
if DO_CHECKS:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], check=True)